In [1]:
# Cell 1 — Imports and load
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

music    = pd.read_csv("../data/raw/Music Info.csv")
listening = pd.read_csv("../data/raw/User Listening History.csv")

print(f"Music    : {music.shape}")
print(f"Listening: {listening.shape}")

Music    : (50683, 21)
Listening: (9711301, 3)


In [2]:
# Cell 2 — Merge datasets
df = listening.merge(music, on='track_id', how='left')

print(f"Merged shape : {df.shape}")
print(f"Sample:")
df.head(3)

Merged shape : (9711301, 23)
Sample:


,track_id,user_id,playcount,name,artist,spotify_preview_url,spotify_id,tags,genre,year,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,TRIRLYL128F42539D1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1,Nothing From Nothing,Billy Preston,https://p.scdn.co/mp3-preview/340c79f3f19b448e...,0d42LKqrWFVu05jAv9mwwE,"soul, funk, piano, 70s, oldies",RnB,2010,...,9,-5.206,0,0.0276,0.585,0.055000,0.331,0.939,102.792,4
1,TRFUPBA128F934F7E1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1,Paper Gangsta,Lady Gaga,https://p.scdn.co/mp3-preview/5d27f0eb10363c13...,079ZOccLpNsrEiKBaiTrqI,"electronic, pop, female_vocalists, dance, pian...",Pop,2012,...,10,-9.344,0,0.0579,0.804,0.782000,0.105,0.800,100.002,4
2,TRLQPQJ128F42AA94F,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1,Soy,Gipsy Kings,https://p.scdn.co/mp3-preview/a103c32c238829f9...,1D3gFdVhv8iJNMTih2Geq3,guitar,NaN,2003,...,5,-11.647,1,0.0673,0.692,0.000002,0.110,0.851,104.518,4


In [3]:
# Cell 3 — Build user feature matrix
# Each row = one user, columns = behavioral + taste features

user_features = df.groupby('user_id').agg(

    # --- Engagement / activity ---
    total_plays      = ('playcount',        'sum'),
    unique_tracks    = ('track_id',         'nunique'),
    unique_artists   = ('artist',           'nunique'),

    # --- Listening depth ---
    # repeat_rate: avg times a track is played — higher = more repetition
    repeat_rate      = ('playcount',        'mean'),

    # --- Music taste: audio features (weighted by playcount) ---
    avg_danceability = ('danceability',     'mean'),
    avg_energy       = ('energy',           'mean'),
    avg_valence      = ('valence',          'mean'),   # 1=happy, 0=sad/tense
    avg_acousticness = ('acousticness',     'mean'),
    avg_instrumentalness = ('instrumentalness', 'mean'),
    avg_speechiness  = ('speechiness',      'mean'),
    avg_liveness     = ('liveness',         'mean'),
    avg_tempo        = ('tempo',            'mean'),
    avg_loudness     = ('loudness',         'mean'),

    # --- Taste breadth ---
    genre_diversity  = ('genre',            'nunique'),  # how many genres explored
    avg_track_year   = ('year',             'mean'),     # preference for old vs new

).reset_index()

print(f"User feature matrix shape: {user_features.shape}")
user_features.head()

User feature matrix shape: (962037, 16)


,user_id,total_plays,unique_tracks,unique_artists,repeat_rate,avg_danceability,avg_energy,avg_valence,avg_acousticness,avg_instrumentalness,avg_speechiness,avg_liveness,avg_tempo,avg_loudness,genre_diversity,avg_track_year
0,00000b722001882066dff9d2da8a775658053ea0,1,1,1,1.000,0.693000,0.803000,0.476000,0.267000,0.000003,0.06190,0.948000,98.804000,-8.076000,1,1998.000000
1,00001638d6189236866af9bbf309ae6c2347ffdc,1,1,1,1.000,0.435000,0.061700,0.071100,0.992000,0.004720,0.06100,0.071700,116.548000,-17.858000,1,2005.000000
2,0000175652312d12576d9e6b84f600caa24c4715,1,1,1,1.000,0.659000,0.958000,0.759000,0.028500,0.000006,0.05610,0.036600,109.058000,-2.975000,1,2014.000000
3,00001cf0dce3fb22b0df0f3a1d9cd21e38385372,13,8,2,1.625,0.559375,0.476625,0.379625,0.378422,0.127705,0.03045,0.151575,103.037500,-9.086125,1,2008.875000
4,0000267bde1b3a70ea75cf2b2d216cb828e3202b,9,9,8,1.000,0.605556,0.827444,0.589667,0.074870,0.669126,0.11930,0.193856,127.920111,-6.203444,1,2006.666667


In [4]:
# Cell 4 — Artist loyalty score
# What % of plays go to the user's single most-played artist?
# High score = loyal to one artist, Low score = diverse taste

artist_loyalty = (
    df.groupby(['user_id', 'artist'])['playcount']
    .sum()
    .reset_index()
)
top_artist_plays = (
    artist_loyalty
    .sort_values('playcount', ascending=False)
    .groupby('user_id')
    .first()
    .reset_index()
    .rename(columns={'playcount': 'top_artist_plays'})
)
user_features = user_features.merge(
    top_artist_plays[['user_id', 'top_artist_plays']], on='user_id', how='left'
)
user_features['artist_loyalty'] = (
    user_features['top_artist_plays'] / user_features['total_plays']
)
user_features.drop(columns=['top_artist_plays'], inplace=True)

print("Artist loyalty added ✓")

Artist loyalty added ✓


In [5]:
# Cell 5 — Preferred genre per user
preferred_genre = (
    df.groupby(['user_id', 'genre'])['playcount']
    .sum()
    .reset_index()
    .sort_values('playcount', ascending=False)
    .groupby('user_id')
    .first()
    .reset_index()
    .rename(columns={'genre': 'preferred_genre'})
)
user_features = user_features.merge(
    preferred_genre[['user_id', 'preferred_genre']], on='user_id', how='left'
)

print("Preferred genre added ✓")
user_features[['user_id', 'preferred_genre']].head(5)

Preferred genre added ✓


,user_id,preferred_genre
0,00000b722001882066dff9d2da8a775658053ea0,Rap
1,00001638d6189236866af9bbf309ae6c2347ffdc,Pop
2,0000175652312d12576d9e6b84f600caa24c4715,Reggae
3,00001cf0dce3fb22b0df0f3a1d9cd21e38385372,Rock
4,0000267bde1b3a70ea75cf2b2d216cb828e3202b,Electronic


In [6]:
# Cell 6 — Create engagement label
# Low engagement = total plays below 25th percentile → label = 1
# High engagement → label = 0

threshold = user_features['total_plays'].quantile(0.25)
user_features['low_engagement'] = (user_features['total_plays'] < threshold).astype(int)

print(f"Threshold           : {threshold:.0f} plays")
print(f"Low engagement (1)  : {user_features['low_engagement'].sum():,}")
print(f"High engagement (0) : {(user_features['low_engagement']==0).sum():,}")
print(f"Low engagement rate : {user_features['low_engagement'].mean()*100:.1f}%")

Threshold           : 5 plays
Low engagement (1)  : 235,486
High engagement (0) : 726,551
Low engagement rate : 24.5%


In [7]:
# Cell 7 — Handle missing values
print("Missing before:")
print(user_features.isnull().sum()[user_features.isnull().sum() > 0])

# fill audio feature nulls with column median
audio_cols = ['avg_danceability', 'avg_energy', 'avg_valence',
              'avg_acousticness', 'avg_instrumentalness', 'avg_speechiness',
              'avg_liveness', 'avg_tempo', 'avg_loudness', 'avg_track_year',
              'genre_diversity', 'artist_loyalty']

for col in audio_cols:
    user_features[col] = user_features[col].fillna(user_features[col].median())

print("\nMissing after:")
print(user_features.isnull().sum()[user_features.isnull().sum() > 0])

Missing before:
preferred_genre    70121
dtype: int64

Missing after:
preferred_genre    70121
dtype: int64


In [8]:
# Cell 8 — Save feature matrix
import os
os.makedirs('../data/processed', exist_ok=True)

user_features.to_csv('../data/processed/user_features.csv', index=False)
print(f"Saved → ../data/processed/user_features.csv")
print(f"Shape : {user_features.shape}")
print(f"\nColumns:")
for col in user_features.columns:
    print(f"  {col}")

Saved → ../data/processed/user_features.csv
Shape : (962037, 19)

Columns:
  user_id
  total_plays
  unique_tracks
  unique_artists
  repeat_rate
  avg_danceability
  avg_energy
  avg_valence
  avg_acousticness
  avg_instrumentalness
  avg_speechiness
  avg_liveness
  avg_tempo
  avg_loudness
  genre_diversity
  avg_track_year
  artist_loyalty
  preferred_genre
  low_engagement
